In [1]:
import numpy as np 
import pandas as pd 
import joblib
from xgboost import XGBClassifier
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

In [2]:
x_train_sm, x_test, y_train_sm, y_test = joblib.load('../data/preprocessed_splits.pkl')
print("Loaded preprocessed data.\nTrain shape: ", x_train_sm.shape)

Loaded preprocessed data.
Train shape:  (454902, 29)


#### Train XGBoost where xgboost is main fraud detector

In [3]:
model = XGBClassifier(
            n_estimators = 100,  # no. of trees
            max_depth = 6,       # depth of each trees 
            random_state = 42,
            eval_metric = "logloss"
        )
model.fit(x_train_sm, y_train_sm)
print("XGBoost training completed.")

XGBoost training completed.


#### Cross Validation

In [4]:
cv_score = cross_val_score(
                model, x_train_sm, y_train_sm, cv=5,
                scoring = "roc_auc",  # fraud metric
                n_jobs = -1           # use all CPU cores so faster
            )

In [5]:
print("CV ROC-AUC scores: ", cv_score)
print("Mean: ", cv_score.mean().round(4))
print("Std: ", cv_score.std().round(4))

CV ROC-AUC scores:  [0.99998241 0.99999856 0.99999266 0.99998252 0.99999525]
Mean:  1.0
Std:  0.0


#### GridSearchCV(Hyperparameter tuning)

In [6]:
df = {"n_estimators":[100,200],
     "max_depth": [4,6],
     "learning_rate": [0.05, 0.1]
     }

grid = GridSearchCV(XGBClassifier(random_state = 42, eval_metric = "logloss"),
                   param_grid= df,
                   cv=3,
                   scoring="roc_auc",
                   n_jobs=-1,
                    verbose=1 
                   )

grid.fit(x_train_sm, y_train_sm)
print("Best params: ", grid.best_params_)
print("Best ROC-AUC: ", grid.best_score_.round(4))

Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best params:  {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200}
Best ROC-AUC:  1.0


#### NOTE:
we found out that the cross validation has been done internally by hyperparameter tuning so no need to do cross validation.

#### Retrain with best parameters

In [15]:
model = grid.best_estimator_
print("Model updated with best parameters.")

Model updated with best parameters.


#### Evaluate XGBoost

In [16]:
y_pred = model.predict(x_test)
y_prob = model.predict_proba(x_test)[:,1] # only fraud prob. selected

print("Classification Report: \n",
      classification_report(y_true=y_test, y_pred=y_pred, target_names=["Legit", "Fraud"]))
print("ROC-AUC Score: ", roc_auc_score(y_test, y_prob))

Classification Report: 
               precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     56864
       Fraud       0.66      0.87      0.75        98

    accuracy                           1.00     56962
   macro avg       0.83      0.93      0.87     56962
weighted avg       1.00      1.00      1.00     56962

ROC-AUC Score:  0.9904781763577688


#### NOTE:
y_prob doesnot reutrn final class label i.e. 0 or 1 rather it return the probability of the class label(0 or 1). eg. 
##### Legit(0)   Fraud(1)
- 0.95        0.05 (it is legit)
- 0.20        0.80 (it is fraud)

ROC-AUC measures how weak the model separates classes across all threshold. A good model gives high fraud prob to actual frauds and low fraud prob to legit transaction because we have selected y_prob as fraud probability. so, high fraud prob for actual frauds

- Precision : Out of all transaction predicted as fraud, how many were actually fraud?
- Recall: Out of all actual frauds, how many did the model catch?
- F1-score: combines both into one number.
- Support: No. of actual samples in that class.
- Accuracy: Overall, how many prediction were correct?
- Macro avg: Treats all classes equally.(it is impt for imbalanced data.)
- Weighted avg: Considers class sizes.

### Train Isolation Forest(unsupervised)

we use isolation forest because it works well with rare anomalies, very few fraud example exits and catches all the new fraud types and is unsupervised model so no y_trian_sm needed

In [17]:
iso = IsolationForest(n_estimators=100, contamination=0.01, random_state=42)
iso.fit(x_train_sm)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",100
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.01
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary `... versionadded:: 0.21",False


In [18]:
iso_preds = iso.predict(x_test)
iso_preds = np.where((iso_preds==-1), 1, 0) 

print(classification_report(y_test, iso_preds, target_names=["Legit", "Fraud"]))

              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     56864
       Fraud       0.25      0.04      0.07        98

    accuracy                           1.00     56962
   macro avg       0.62      0.52      0.53     56962
weighted avg       1.00      1.00      1.00     56962



### Note: 
iso_preds ouptut was 1 for normal point(inlier) and -1 for anomaly(outlier)
so np.where is used to transform -1 to 1 (anomaly) and 1 to 0(normal).

np.where(condition, value_if_true, value_if_false)

#### Why this conversion is useful
Many evaluation metrics in scikit-learn expect:
1 = positive class
0 = negative class
So this mapping makes it easier to compute:
accuracy, precision, recall, F1-score, confusion matrix

In [19]:
joblib.dump(model, '../models/fraud_model.pkl')
print("Model saved to ../models/fraud_model.pkl")

Model saved to ../models/fraud_model.pkl


## Summary
- Loaded preprocessed splits from Phase 3
- Trained XGBoost (main model) expect ROC-AUC above 0.95
- Trained Isolation Forest (anomaly detection, no labels needed)
- XGBoost will significantly outperform Isolation Forest here
- Saved model to models/fraud_model.pkl for Phase 6 (FastAPI)